In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
os.chdir("/Users/kanhapandey/Projects/SIH_PAIMANA")

In [ ]:
df = pd.read_csv("data/raw/quaterly_ongoing_projects.csv")

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df

In [ ]:
df.drop(1738, inplace=True)
df

In [ ]:
df.drop("Sl No", axis=1, inplace=True)
df

In [ ]:
def split_name_agency_code(text: str):
    if pd.isna(text):
        return pd.Series([None, None, None])

    text = text.replace('\n', ' ').strip()

    # -------------------------
    # Getting project code
    # -------------------------
    c_close_idx = text.rfind(')')
    c_balance = 0
    c_open_idx = None

    for i in range(c_close_idx, -1, -1):
        c_ch = text[i]

        if c_ch == ')':
            c_balance += 1
        elif c_ch == '(':
            c_balance -= 1

            if c_balance == 0:
                c_open_idx = i
                break

    code = text[c_open_idx + 1:c_close_idx].strip()

    # Remaining text contains project name + agency
    name_agency = text[:c_open_idx].strip()

    # -------------------------
    # Getting agency name
    # -------------------------
    a_close_idx = name_agency.rfind(')')
    a_balance = 0
    a_open_idx = None

    for i in range(a_close_idx, -1, -1):
        a_ch = name_agency[i]

        if a_ch == ')':
            a_balance += 1
        elif a_ch == '(':
            a_balance -= 1

            if a_balance == 0:
                a_open_idx = i
                break

    agency = name_agency[a_open_idx + 1:a_close_idx].strip()

    # Remaining text is project name
    project_name = name_agency[:a_open_idx].strip()

    return pd.Series([project_name, agency, code])

In [ ]:
df[["project_name", "agency", "project_code"]] = (
    df["Project Name (Agency Name) (Project Code)"].apply(split_name_agency_code)
)
df.head()

In [ ]:
df.drop("Project Name (Agency Name) (Project Code)", axis=1, inplace=True)
df.head()

In [ ]:
parts = df['Date of Commissioning Original (Revised) {Anticipated} (MM/YYYY)'].str.replace(
    r'[)}]', '', regex=True
).str.split(r'[({]', regex=True, expand=True)

df['date_of_commissioning'] = parts[0].str.strip().replace('', np.nan)
df['revised_date'] = parts[1].str.strip().replace('', np.nan)
df['anitcipated_date'] = parts[2].str.strip().replace('', np.nan)

In [ ]:
df.head()

In [ ]:
df.drop("Date of Commissioning Original (Revised) {Anticipated} (MM/YYYY)", axis =1, inplace=True)
df.head()

In [ ]:
parts =df["Cost Original (Revised) {Anticipated} in Rs. Crore"].str.replace(r'[)}]', '', regex=True).str.split(r'[({]',regex=True, expand=True)
df["original_cost_cr"]= parts[0].str.strip()
df["revised_cost_cr"] = parts[1].str.strip()
df["anticipated_cost_cr"] = parts[2].str.strip()
df.drop("Cost Original (Revised) {Anticipated} in Rs. Crore", axis=1, inplace=True)
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
date_cols = ["Date of Approval (MM/YYYY)", "date_of_commissioning", "anitcipated_date", "revised_date"]
missing_tokens = {'-', 'NA', 'na', 'N/A', ''}

#for c in date_cols:
#    df[c] = df[c].replace(missing_tokens, np.nan)
#    df[c] = pd.to_datetime(df[c], format='%m/%Y', errors='coerce')

In [ ]:
cost_cols = ["Cumulative Expenditure in Rs. Crore", "original_cost_cr", "revised_cost_cr", "anticipated_cost_cr"]
for c in cost_cols:
    df[c] = (df[c].replace(missing_tokens, np.nan).str.replace(",", "", regex=False))
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
for c in date_cols:
    print("\n", c)
    print(df[c].dropna().head(10).tolist())

In [ ]:
for c in cost_cols:
    print("\n", c)
    print(df[c].dropna().head(10).tolist())